# Teste de estresse de drift do modelo de crédito

## Ampliação: bateria sistemática executada

Os notebooks 04–08 ampliam os experimentos iniciais. Cada célula contém a explicação do método, o código, os resultados e gráficos. Os CSVs preservam cada cenário para permitir novos cortes e critérios.

| Notebook | Experimento | Arquivo de cenários |
|---|---|---|
| [04_univariado_extensivo.ipynb](04_univariado_extensivo.ipynb) | **2.560** mudanças de composição de valores (10 variáveis × 8 trajetórias × 32 intensidades), cada uma avaliada em **todas** as métricas; mais 128 mudanças da composição de faltantes existentes. Fronteiras por métrica, PSI/Wasserstein/KS e bootstrap. | [resultados_univariado_extensivo.csv](resultados_univariado_extensivo.csv), [fronteiras_univariado_extensivo.csv](fronteiras_univariado_extensivo.csv) |
| [05_multivariado_populacional_extensivo.ipynb](05_multivariado_populacional_extensivo.ipynb) | **3.072** cenários rotulados: 256 direções Sobol × 12 intensidades em quatro famílias (caudas conjuntas, interação entre pares, mistura de grupos e postos esparsos). Repondera linhas completas, preservando suas combinações X–Y; seleção e confirmação em metades separadas. | [resultados_multivariado_populacional.csv](resultados_multivariado_populacional.csv) |
| [06_multivariado_sintetico_extensivo.ipynb](06_multivariado_sintetico_extensivo.ipynb) | **2.048** intervenções internas via cópula de postos com propagação condicional e truncamento nos mínimos/máximos observados no treino; **128** intervenções externas sem truncamento; oito cenários confirmados nas 101.503 linhas de `cs-test.csv`. Inclui Morris com SALib. | [resultados_multivariado_sintetico.csv](resultados_multivariado_sintetico.csv) |
| [07_drift_observado_importance_weighting.ipynb](07_drift_observado_importance_weighting.ipynb) | Classificador de domínio e razão de densidades para comparar holdout ao `cs-test.csv` real, com incerteza e ESS. | [resultados_drift_observado.csv](resultados_drift_observado.csv) |
| [08_dro_grupos.ipynb](08_dro_grupos.ipynb) | **245** intensidades de pior mistura de 20 grupos definidos só por X, com seleção e confirmação separadas e bootstrap. | [resultados_dro_grupos.csv](resultados_dro_grupos.csv) |

**Leitura principal.** Uma fronteira de drift depende da direção da mudança, da métrica, do corte e do limite adotado; não há um único PSI que assegure desempenho. A primeira falha univariada da grade principal ocorreu em menor frequência de atraso de 90 dias (PSI ≈ 0,0057, por recall), mas seu intervalo bootstrap cruza o limite ilustrativo. Na varredura multivariada, a primeira falha selecionada para interação de par ocorreu na intensidade 0,09, após 0,06 aprovado na exploração; a confirmação falhou já em 0,06. Seu PSI dos 16 grupos foi **0,000222** e o PSI bivariado do par **0,000455**. Mesmo diagnósticos conjuntos podem ser pequenos quando uma métrica sensível ao corte sofre mudança. As quatro famílias, trajetórias e limites estão tabulados no notebook 05.

**Relações e limites.** Reponderar linhas completas nos notebooks 04 e 05 mantém exatamente as combinações observadas e permite calcular AUC, AP, Brier, recall e FPR sob a hipótese de estabilidade de `P(Y|X)`. A reponderação não cria valores novos; algumas linhas originais do holdout já podem exceder os extremos vistos no treino. Truncá-las alteraria X sem um novo rótulo válido. Para testar **valores novos** com truncamento e extrapolação, use 06: sua cópula preserva aproximadamente dependências de postos, mantém as contagens inteiras e ausências, e registra correlações e violações de domínio. Dos 2.048 cenários internos, 1.710 tiveram diferença média de correlação de postos ≤0,10; **1.517** também não violaram o domínio básico auditado, incluindo idade ≥18. A coluna `viavel_diagnostico` permite filtrá-los. São diagnósticos de plausibilidade, não garantias causais ou de regras de crédito.

**Sensibilidade de previsão.** No notebook 06, 801 cenários internos respeitaram os limites ilustrativos de até 5% de decisões invertidas e até 0,02 de mudança absoluta no score médio; **673** deles também passaram pelo filtro diagnóstico de plausibilidade. Isso mede **score e decisão**, não perda de acerto: `cs-test.csv` não tem rótulos, e o rótulo antigo de uma linha transformada não é um contrafactual válido. As fronteiras de desempenho rotulado vêm de 04 e 05. Os 128 cenários externos são testes de extrapolação e validação de entrada; intervenções abaixo de mínimos zero podem produzir valores negativos deliberadamente implausíveis.

**Drift observado e pior mistura.** O classificador do notebook 07 distinguiu holdout de `cs-test.csv` com AUC **0,5004** (IC bootstrap 95%: **0,4938–0,5066**). A AUC reponderada do holdout foi **0,86524**, contra **0,86470** sem pesos, com ESS de 99,93%. É uma estimativa dependente de `P(Y|X)` estável, sem prova de desempenho no arquivo sem rótulos. No notebook 08, o critério composto falhou em θ=2,26 no desenvolvimento (KL≈0,0087; PSI≈0,0167) e θ=2,08 na confirmação (KL≈0,0073; PSI≈0,0140), por FPR. Esse limite vale para a família de misturas de 20 grupos testada, não para toda distribuição futura.

**Critérios e execução.** Os guardrails dos testes rotulados são ilustrativos e editáveis: queda de AUC ≤0,02; AP/prevalência ≥85% do basal; Brier ≤115% do basal; queda de recall ≤0,05; aumento de FPR ≤0,01; ESS ≥35% do nominal. O corte é 0,50. O notebook 06 usa os mínimos e máximos **da fração de treino do modelo**, pois não foram fornecidos limites de negócio; o mínimo histórico de idade pode ser inadequado, motivo da auditoria de domínio. Os notebooks 04–08 carregam [model_class_retreinado.json](model_class_retreinado.json) sem reajustar o modelo. O treinamento idêntico no 00 requer CUDA. Substitua os critérios e limites por regras aprovadas para o uso real antes de interpretar as fronteiras operacionalmente.

Referências adicionais: [Namkoong e Duchi (2016), DRO](https://papers.nips.cc/paper_files/paper/2016/hash/4588e674d3f0faf985047d4c3f13ed0d-Abstract.html), [SciPy Sobol](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.qmc.Sobol.html), [SALib Morris](https://salib.readthedocs.io/en/latest/api.html).

## Ordem de uso

Execute os notebooks nesta pasta, nesta ordem:

1. [`00_preparacao_modelo.ipynb`](00_preparacao_modelo.ipynb): reconstrói o `model_class`, valida os arquivos e estabelece a linha de base.
2. [`01_drift_univariado.ipynb`](01_drift_univariado.ipynb): repondera linhas completas em duas direções para cada uma das 10 variáveis; testa ausência adicional de dados por variável e em conjunto.
3. [`02_drift_multivariado.ipynb`](02_drift_multivariado.ipynb): repondera sete subgrupos definidos por duas condições simultâneas e diagnostica a diferença conjunta entre `cs-training.csv` e `cs-test.csv`.
4. [`03_sensibilidade_sintetica.ipynb`](03_sensibilidade_sintetica.ipynb): altera sinteticamente a utilização de crédito e propaga parte da mudança a dívida e idade para medir movimento de score e decisões.

Os caminhos dos CSVs estão na primeira célula de configuração de cada notebook e podem ser ajustados. O notebook 00 requer CUDA, como o treinamento original. Os demais carregam `model_class_retreinado.json` e não ajustam o modelo. Os CSVs de resultados contêm todas as linhas de cenário; os gráficos e explicações estão embutidos nos notebooks.

## O que é possível concluir com estes dados

`cs-training.csv` tem 150.000 observações rotuladas. O split original produz 104.996 linhas de treino, 22.504 de validação e um holdout de 22.500 linhas. A tolerância de desempenho é estimada **somente nesse holdout rotulado**, que não foi usado para ajustar as árvores.

`cs-test.csv` tem 101.503 linhas e todos os valores de `SeriousDlqin2yrs` ausentes. Ele permite medir drift de entradas e scores, mas não AUC, AP, Brier, recall ou tolerância de desempenho. O notebook original não serializa o Booster `model_class`; o JSON entregue é uma **reconstrução** com os mesmos parâmetros, split e 150 árvores efetivamente treinadas (melhor iteração 119). A linha de base ficou próxima da exibida no original:

| Holdout rotulado | Valor |
|---|---:|
| ROC AUC | 0,8647 |
| Average precision | 0,3942 |
| Brier | 0,0527 |
| Recall no corte 0,50 | 0,3200 |
| Prevalência observada | 0,0667 |
| Score médio | 0,1102 |

O score médio supera a prevalência observada. A calibração da linha de base deve ser examinada antes de usar scores como probabilidades de inadimplência.

## Definição ilustrativa de tolerância

Os notebooks 01 e 02 usam os **mesmos critérios editáveis**, escolhidos apenas para demonstrar o método: queda de ROC AUC até 0,02; `AP / prevalência` pelo menos 85% do basal; Brier até 115% do basal; queda de recall até 0,05; aumento de FPR até 0,01; tamanho efetivo dos pesos (`ESS`) pelo menos 35% do tamanho nominal. A fronteira é a última intensidade aprovada **antes da primeira falha na grade testada**. Ela depende do cenário, da direção, do corte 0,50 e dos critérios. Intervalos bootstrap pareados de 95% ajudam a avaliar a incerteza; a fronteira apresentada usa estimativas pontuais.

Exemplos desta execução:

| Cenário | Último nível aprovado | Primeiro nível reprovado | Motivo principal |
|---|---:|---:|---|
| Cauda alta de `RevolvingUtilizationOfUnsecuredLines` | força exponencial 0,25, PSI 0,0205 | 0,50, PSI 0,0811 | AP/lift, Brier e FPR |
| Ausência adicional apenas em `RevolvingUtilizationOfUnsecuredLines` | 20% dos valores presentes apagados | 30% | AUC e Brier |
| Utilização ≥0,75 **e** dívida ≥0,5 | razão de pesos 2, PSI conjunto 0,0415 | 3 | Brier |
| Atrasos de 90 dias **e** de 60–89 dias | razão de pesos 2, PSI conjunto 0,0124 | 3 | FPR |

Nas demais variáveis, direções e combinações há outras fronteiras; consulte as tabelas completas e os gráficos. **Força exponencial univariada e razão de pesos multivariada são escalas diferentes.** O PSI univariado usa bins da própria variável; o PSI conjunto usa quatro células de duas condições. Portanto, os seus valores não são equivalentes entre metodologias e não constituem um único “PSI seguro”.

A diferença observada entre `cs-training.csv` e `cs-test.csv` foi pequena nesta execução: PSI marginal abaixo de 0,0003 por variável, PSI do score de cerca de 0,00026, e AUC do classificador de domínio 0,5021 (IC 95% 0,4940–0,5103). Isso não demonstra que o desempenho no arquivo sem rótulos seja igual ao do holdout.

No teste sintético mais forte de aumento da utilização (intensidade 0,7), o movimento absoluto médio do score foi 3,62 pontos percentuais sem propagação e 4,31 com propagação total para dívida e idade. Respectivamente 2,13% e 2,65% das decisões cruzaram o corte 0,50. São medidas de **sensibilidade de previsão**, não de acerto.

## Como transformar em monitoramento

Antes de definir alertas de produção, substitua os critérios ilustrativos por limites aprovados para o uso do modelo e reproduza os cenários em uma amostra temporal rotulada representativa. Monitore por variável e direção, qualidade/ausência, score e combinações relevantes; confirme perda de desempenho quando os rótulos chegarem. Reponderação de linhas observadas pressupõe estabilidade de `P(Y|X)` dentro do suporte histórico. Mudança da relação entre perfil e inadimplência, novas combinações de características e viés de rótulos por política de aprovação exigem avaliação adicional.

## Referências metodológicas

- [Sugiyama et al. (2007), covariate shift e ponderação](https://www.jmlr.org/papers/v8/sugiyama07a.html)
- [Subbaswamy et al. (2021), robustez a mudanças de distribuição](https://proceedings.mlr.press/v130/subbaswamy21a.html)
- [Lopez-Paz e Oquab (2018), teste de duas amostras por classificador](https://arxiv.org/abs/1610.06545)
- [Documentação do XGBoost para `train` e early stopping](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.train)
